In [2]:
import re
import pandas as pd
import numpy as np

df = pd.read_csv("DataCo_Supply_Chain_Dataset.csv", encoding="latin-1")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()

Rows: 180519
Columns: 53


,ï»¿Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   ï»¿Type                        180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname                 180519 non

In [4]:
# Clean Column Names for SQL
df.columns = [
    re.sub(r"[^\w]+", "_", c.strip().lower()).strip("_")
    for c in df.columns
]

# Fix Encoding Issue
if "ï_type" in df.columns:
    df.rename(columns={"ï_type": "type"}, inplace=True)

print("First 10 Columns:")
print(df.columns[:10].tolist())

First 10 Columns:
['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city']


In [5]:
# Check Data Types
print(df.dtypes)

type                               str
days_for_shipping_real           int64
days_for_shipment_scheduled      int64
benefit_per_order              float64
sales_per_customer             float64
delivery_status                    str
late_delivery_risk               int64
category_id                      int64
category_name                      str
customer_city                      str
customer_country                   str
customer_email                     str
customer_fname                     str
customer_id                      int64
customer_lname                     str
customer_password                  str
customer_segment                   str
customer_state                     str
customer_street                    str
customer_zipcode               float64
department_id                    int64
department_name                    str
latitude                       float64
longitude                      float64
market                             str
order_city               

In [6]:
# Check Missing Values
missing = df.isnull().sum()

print("Missing Values:")
print(missing[missing > 0])

Missing Values:
customer_lname              8
customer_zipcode            3
order_zipcode          155679
product_description    180519
dtype: int64


In [7]:
# Remove Duplicates
print("Duplicate Rows:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print("Rows After Removing Duplicates:", len(df))

Duplicate Rows: 0
Rows After Removing Duplicates: 180519


In [8]:
# Handle Missing Values
df["customer_zipcode"] = (
    pd.to_numeric(df["customer_zipcode"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .fillna("Unknown")
)

df["customer_lname"] = df["customer_lname"].fillna("Unknown")

df = df.drop(
    columns=["product_description", "order_zipcode"],
    errors="ignore"
)

print("Remaining Missing Values:", df.isnull().sum().sum())

Remaining Missing Values: 0


In [9]:
# Fix Data Types
date_columns = [
    "order_date_dateorders",
    "shipping_date_dateorders"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(df[date_columns].dtypes)

order_date_dateorders       datetime64[us]
shipping_date_dateorders    datetime64[us]
dtype: object


In [10]:
# Clean Text Data
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

print("Text data cleaned successfully.")

Text data cleaned successfully.


In [11]:
# Final Data Check
print("Final Shape:", df.shape)
print("Missing Values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())

Final Shape: (180519, 51)
Missing Values: 0
Duplicates: 0


In [12]:
# Export Cleaned Dataset
output_file = "DataCo_Supply_Chain_Cleaned.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d %H:%M:%S"
)

print("Cleaned Dataset Saved Successfully:", output_file)

Cleaned Dataset Saved Successfully: DataCo_Supply_Chain_Cleaned.csv


In [13]:
cleaned_df = pd.read_csv(
    output_file,
    parse_dates=date_columns,
    low_memory=False
)

print("Reloaded Shape:", cleaned_df.shape)
print("Duplicates:", cleaned_df.duplicated().sum())
print("Missing Values:", cleaned_df.isnull().sum().sum())

cleaned_df.head()

Reloaded Shape: (180519, 51)
Duplicates: 0
Missing Values: 0


,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,order_state,order_status,product_card_id,product_category_id,product_image,product_name,product_price,product_status,shipping_date_dateorders,shipping_mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,Java Occidental,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,RajastÎ±n,PENDING,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,RajastÎ±n,CLOSED,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,Queensland,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,Queensland,PENDING_PAYMENT,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class
